#Initialisations

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, add_months, substring, to_date, expr
from pyspark.sql.types import StringType

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_key",
    "sls_order_dt": "order_date",
    "sls_cust_id": "customer_id",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales",
    "sls_quantity" : "quantity",
    "sls_price" : "price"
}

#Read the bronze table in

In [0]:
df = spark.read.table("bronze.crm_sales_details_raw")

#Transformations

##1. Trimming all the text columns

In [0]:
for field in df.schema:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##2. Fixing the date columns: formart and Dataype

In [0]:
df = df.withColumn(
    "sls_order_dt",
    expr("try_to_date(cast(sls_order_dt as string), 'yyyyMMdd')")
)

df = df.withColumn(
    "sls_ship_dt",
    expr("try_to_date(cast(sls_ship_dt as string), 'yyyyMMdd')")
)

df = df.withColumn(
    "sls_due_dt",
    expr("try_to_date(cast(sls_due_dt as string), 'yyyyMMdd')")
)

##3. Renaming columns for better readability

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Writting to the silver layer

In [0]:
(
    df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.crm_sales")
)